In [ ]:
import plotly.express as px
import matplotlib as mpl
import matplotlib.pyplot as plt
from hsi_detect.spectrum import Spectrum
from hsi_detect.image import HyperspectralImage
from hsi_detect.classifier import HierarchicalKMeansUnmixer
from hsi_detect.utils import *
from datetime import date
import os 

today = date.today()
date_str = today.strftime('%d%b%Y')
print ('Date prefix:', date_str)

# Plotting parameters
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['lines.linewidth'] = 0.5
mpl.rcParams['axes.linewidth']= 0.5
mpl.rcParams['xtick.major.width'] = 0.5
mpl.rcParams['xtick.minor.width'] = 0.5
mpl.rcParams['ytick.major.width'] = 0.5
mpl.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.size'] = 7

In [ ]:
IMAGE_PATH = '00_data/rg_yf10_gradient_varied_soils/data.hdr'
REFERENCE_SPECTRUM_PATH = '00_data/absorbance_data/YF10_infered_absorbance_from_pellets_09Jul2024.npy'

savedir = '/'.join(IMAGE_PATH.split('/')[:-1])+f'{IMAGE_PATH.split("/")[-1].split(".hdr")[0]}_outputs_from_analysis/'

print (savedir)
if not os.path.isdir(savedir):
    os.mkdir(savedir)
    print ('Made directory:', savedir)

In [ ]:
# Load and visualize image
hsi_img = HyperspectralImage(IMAGE_PATH, smoothing_window=11)
hsi_img.show(dpi=300, savepath=savedir+f'{date_str}_reconstructed_RGB.png')

# Load and visualize the spectrum of the HSR
reference_spectrum = Spectrum('00_data/absorbance_data/YF10_infered_absorbance_from_pellets_09Jul2024.npy')
reference_spectrum.interpolate_spectrum(hsi_img.centers) #Interpolate spectrum to fit the HSI
reference_spectrum.show()

In [ ]:
hsi_img.image = hsi_img.image[:,100:,:]
hsi_img.rgb = hsi_img.rgb[:,100:,:]
hsi_classifier = HierarchicalKMeansUnmixer()
hsi_classifier.fit(hsi_img, reference_spectrum)
scored_img = hsi_classifier.classify(reference_spectrum)

In [ ]:
# The clustering results can be inspected
hsi_classifier.visualize_clusters()
hsi_classifier.visualize_endmembers()

In [ ]:
# Visualize classified image
plt.figure(dpi=500)
plt.imshow(scored_img, vmin=0.0, vmax=0.2, cmap='inferno')
plt.xticks([])
plt.yticks([])
plt.box(False)
plt.show()

In [ ]:
html = make_juxtaposed_html(hsi_img.rgb.astype(np.uint8), (scored_img*255).astype(np.uint8))

with open(f'{savedir}/juxtaposed_rgb-classified_images.html', 'w') as f:
  f.write(html)